# Leaderboard Analysis

This notebook shows how to:
- Analyze leaderboard submissions
- Compare submissions across suites
- Identify top performers
- Visualize leaderboard trends

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from agentick.leaderboard.suites import OFFICIAL_SUITES, list_suites

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## List Available Suites

First, let's see what benchmark suites are available:

In [ ]:
# Get all suites
suites = list_suites()

print(f"Total Benchmark Suites: {len(suites)}\n")
print("Available Suites:")
for i, suite_name in enumerate(suites, 1):
    suite = OFFICIAL_SUITES[suite_name]
    print(f"  {i}. {suite_name}")
    print(f"     Tasks: {len(suite.tasks)}")
    print(f"     Description: {suite.description[:60]}...")
    print()

## Create Sample Submissions

Let's create and evaluate some sample submissions:

In [ ]:
from agentick.leaderboard.submission import SubmissionSpec

# Create sample submission specs
submissions = [
    SubmissionSpec(
        agent_name="Random-Baseline-v1",
        author="Demo Author",
        description="Random baseline agent",
        agent_type="code",
        observation_mode="ascii",
        config={"type": "random"},
        suites=["agentick-core-v1"],
    ),
    SubmissionSpec(
        agent_name="Oracle-Reference-v1",
        author="Demo Author",
        description="Oracle reference agent",
        agent_type="code",
        observation_mode="ascii",
        config={"type": "oracle"},
        suites=["agentick-core-v1"],
    ),
]

print(f"Created {len(submissions)} sample submissions")
for sub in submissions:
    print(f"  - {sub.agent_name}")

## Simulated Leaderboard Data

For this demo, we'll create simulated leaderboard data:

In [ ]:
# Simulate leaderboard results
leaderboard_data = [
    {
        "Agent": "GPT-4o-Vision",
        "Success Rate": 0.92,
        "Mean Return": 0.88,
        "Mean Length": 15.2,
        "Suite": "Core",
    },
    {
        "Agent": "Claude-3.5-Vision",
        "Success Rate": 0.89,
        "Mean Return": 0.85,
        "Mean Length": 16.1,
        "Suite": "Core",
    },
    {
        "Agent": "PPO-Trained",
        "Success Rate": 0.75,
        "Mean Return": 0.71,
        "Mean Length": 22.5,
        "Suite": "Core",
    },
    {
        "Agent": "DQN-Trained",
        "Success Rate": 0.68,
        "Mean Return": 0.65,
        "Mean Length": 25.3,
        "Suite": "Core",
    },
    {
        "Agent": "Oracle",
        "Success Rate": 0.98,
        "Mean Return": 0.95,
        "Mean Length": 12.1,
        "Suite": "Core",
    },
    {
        "Agent": "Random",
        "Success Rate": 0.12,
        "Mean Return": 0.08,
        "Mean Length": 48.7,
        "Suite": "Core",
    },
    {
        "Agent": "Greedy",
        "Success Rate": 0.45,
        "Mean Return": 0.42,
        "Mean Length": 32.4,
        "Suite": "Core",
    },
]

df = pd.DataFrame(leaderboard_data)
df_sorted = df.sort_values("Success Rate", ascending=False).reset_index(drop=True)
df_sorted.index += 1  # Start ranking from 1

print("\nLeaderboard Rankings (Core Suite):")
print("=" * 70)
print(df_sorted.to_string())

## Visualize Leaderboard

In [ ]:
# Create leaderboard visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Success rate ranking
agents = df_sorted["Agent"].tolist()
success_rates = df_sorted["Success Rate"].tolist()
colors = plt.cm.viridis(np.linspace(0, 1, len(agents)))

y_pos = np.arange(len(agents))
ax1.barh(y_pos, success_rates, color=colors, alpha=0.8)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(agents)
ax1.invert_yaxis()  # Top to bottom
ax1.set_xlabel("Success Rate", fontsize=12)
ax1.set_title("Leaderboard Rankings by Success Rate", fontsize=14, fontweight="bold")
ax1.set_xlim(0, 1)
ax1.grid(axis="x", alpha=0.3)

# Add value labels
for i, (agent, sr) in enumerate(zip(agents, success_rates)):
    ax1.text(sr + 0.02, i, f"{sr:.2%}", va="center", fontsize=10)

# Success rate vs Episode length scatter
success_rates_all = df["Success Rate"].tolist()
mean_lengths = df["Mean Length"].tolist()
agents_all = df["Agent"].tolist()

scatter = ax2.scatter(
    mean_lengths,
    success_rates_all,
    s=200,
    c=success_rates_all,
    cmap="RdYlGn",
    alpha=0.7,
    edgecolors="black",
    linewidth=1.5,
)
ax2.set_xlabel("Mean Episode Length", fontsize=12)
ax2.set_ylabel("Success Rate", fontsize=12)
ax2.set_title("Success Rate vs Episode Length", fontsize=14, fontweight="bold")
ax2.grid(True, alpha=0.3)

# Add agent labels
for agent, length, sr in zip(agents_all, mean_lengths, success_rates_all):
    ax2.annotate(agent, (length, sr), fontsize=8, ha="center", va="bottom")

plt.colorbar(scatter, ax=ax2, label="Success Rate")
plt.tight_layout()
plt.show()

## Performance Tiers

Group agents into performance tiers:

In [ ]:
def classify_tier(success_rate):
    """Classify agent into performance tier."""
    if success_rate >= 0.9:
        return "S-Tier (Excellent)"
    elif success_rate >= 0.7:
        return "A-Tier (Strong)"
    elif success_rate >= 0.5:
        return "B-Tier (Good)"
    elif success_rate >= 0.3:
        return "C-Tier (Fair)"
    else:
        return "D-Tier (Weak)"


df["Tier"] = df["Success Rate"].apply(classify_tier)

print("\nPerformance Tiers:")
print("=" * 60)
for tier in [
    "S-Tier (Excellent)",
    "A-Tier (Strong)",
    "B-Tier (Good)",
    "C-Tier (Fair)",
    "D-Tier (Weak)",
]:
    tier_agents = df[df["Tier"] == tier]["Agent"].tolist()
    if tier_agents:
        print(f"\n{tier}:")
        for agent in tier_agents:
            sr = df[df["Agent"] == agent]["Success Rate"].values[0]
            print(f"  - {agent:20s} ({sr:.2%})")

## Statistical Analysis

Compute statistics across the leaderboard:

In [ ]:
# Exclude oracle and random for fair comparison
df_fair = df[~df["Agent"].isin(["Oracle", "Random"])]

print("\nLeaderboard Statistics (excluding Oracle & Random):")
print("=" * 60)
print(f"Number of agents: {len(df_fair)}")
print("\nSuccess Rate:")
print(f"  Mean: {df_fair['Success Rate'].mean():.2%}")
print(f"  Std:  {df_fair['Success Rate'].std():.2%}")
print(f"  Min:  {df_fair['Success Rate'].min():.2%}")
print(f"  Max:  {df_fair['Success Rate'].max():.2%}")
print("\nMean Return:")
print(f"  Mean: {df_fair['Mean Return'].mean():.3f}")
print(f"  Std:  {df_fair['Mean Return'].std():.3f}")
print("\nMean Episode Length:")
print(f"  Mean: {df_fair['Mean Length'].mean():.1f}")
print(f"  Std:  {df_fair['Mean Length'].std():.1f}")

## Gap Analysis

Analyze the gap between current agents and oracle:

In [ ]:
oracle_sr = df[df["Agent"] == "Oracle"]["Success Rate"].values[0]
best_agent_sr = df_fair["Success Rate"].max()
best_agent = df_fair[df_fair["Success Rate"] == best_agent_sr]["Agent"].values[0]

gap = oracle_sr - best_agent_sr
gap_percent = (gap / oracle_sr) * 100

print("\nGap to Oracle Performance:")
print("=" * 60)
print(f"Oracle success rate: {oracle_sr:.2%}")
print(f"Best agent ({best_agent}): {best_agent_sr:.2%}")
print(f"Absolute gap: {gap:.2%}")
print(f"Relative gap: {gap_percent:.1f}% of oracle performance")
print(f"\nRoom for improvement: {gap_percent:.1f}%")

In [ ]:
# Visualize gap to oracle
fig, ax = plt.subplots(figsize=(12, 6))

agents_sorted = df_sorted["Agent"].tolist()
success_sorted = df_sorted["Success Rate"].tolist()

# Create bar chart
colors_gap = ["gold" if agent == "Oracle" else "steelblue" for agent in agents_sorted]
bars = ax.bar(agents_sorted, success_sorted, color=colors_gap, alpha=0.8)

# Add oracle reference line
ax.axhline(y=oracle_sr, color="red", linestyle="--", linewidth=2, label=f"Oracle ({oracle_sr:.2%})")

ax.set_ylabel("Success Rate", fontsize=12)
ax.set_title("Performance Gap to Oracle", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
ax.legend()
plt.xticks(rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Export Leaderboard

Save leaderboard data:

In [ ]:
# Export to CSV
output_dir = Path("results/leaderboard")
output_dir.mkdir(parents=True, exist_ok=True)

leaderboard_file = output_dir / "leaderboard_rankings.csv"
df_sorted.to_csv(leaderboard_file)
print(f"\nLeaderboard saved to: {leaderboard_file}")

# Also save tier classification
tier_file = output_dir / "performance_tiers.csv"
df[["Agent", "Success Rate", "Tier"]].sort_values("Success Rate", ascending=False).to_csv(
    tier_file, index=False
)
print(f"Tier classification saved to: {tier_file}")

## Insights and Recommendations

Key takeaways from the leaderboard:

In [ ]:
print("\n" + "=" * 70)
print("KEY INSIGHTS")
print("=" * 70)

print(f"\n1. Top Performer: {best_agent}")
print(f"   Success rate: {best_agent_sr:.2%}")

print("\n2. Performance Distribution:")
for tier in ["S-Tier (Excellent)", "A-Tier (Strong)", "B-Tier (Good)"]:
    count = len(df[df["Tier"] == tier])
    if count > 0:
        print(f"   {tier}: {count} agent(s)")

print("\n3. Improvement Opportunities:")
print(f"   Gap to oracle: {gap_percent:.1f}%")
print(f"   Average success rate: {df_fair['Success Rate'].mean():.2%}")

print("\n4. Recommended Next Steps:")
if gap_percent > 20:
    print("   - Significant room for improvement exists")
    print("   - Focus on learning from oracle demonstrations")
    print("   - Consider hybrid approaches (RL + LLM)")
else:
    print("   - Agents approaching optimal performance")
    print("   - Focus on efficiency and generalization")

print("\n" + "=" * 70)

## Next Steps

- Train your own agents
- Submit to the leaderboard
- Analyze specific task performance
- Study top-performing agent approaches
- See **05_custom_task.ipynb** for creating custom tasks